# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muneeb-th/ML-Assignment-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Muneeb-th/ML-Assignment-1/main/data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Rebuild Week 4 baseline score
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

print(f"Loaded {len(df)} rows. Base rate: {df['is_declining'].mean():.3f}")

Loaded 30000 rows. Base rate: 0.542


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is "which first?" ranking, so per the skill guide I'm using a
classifier's predicted probability evaluated at Precision@K — same
approach as Week 1/2. I'll use a Random Forest since it captured more
signal than a single decision tree in Week 1, while still checking
feature importances for a sanity check against leakage.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Using client-holdout validation: whole clients are kept out of training
so the model is tested on clients it never saw. A random row split would
let the model implicitly memorize per-client patterns, since pages from
the same client can share characteristics — that would inflate the score
without proving real generalization.

In [7]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["impressions_90d", "sessions_90d", "content_age_days",
                 "days_since_last_update", "ctr", "avg_position", "word_count"]

model_data = df.dropna(subset=feature_cols + ["client_id"])
X = model_data[feature_cols]
y = model_data["is_declining"]
groups = model_data["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
baseline_te = model_data["baseline_score"].iloc[test_idx]

print(f"Train: {len(X_tr)} rows, {model_data['client_id'].iloc[train_idx].nunique()} clients")
print(f"Test: {len(X_te)} rows, {model_data['client_id'].iloc[test_idx].nunique()} clients")

Train: 16271 rows, 24 clients
Test: 6030 rows, 8 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced",
                                random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
model_scores_te = model.predict_proba(X_te)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Comparison table: baseline vs model (same test split, same metric)\n")
print(f"{'Method':<12} {'Precision@20':>14} {'Precision@50':>14}")
for k in (20, 50):
    b = precision_at_k(baseline_te.values, y_te.values, k)
    m = precision_at_k(model_scores_te, y_te.values, k)
    print(f"{'baseline':<12} {b:>14.3f} {'':>14}") if k == 20 else None

for label, scores in [("baseline", baseline_te.values), ("model", model_scores_te)]:
    row = f"{label:<12}"
    for k in (20, 50):
        p = precision_at_k(scores, y_te.values, k)
        row += f" {p:>14.3f}"
    print(row)

base_rate_test = y_te.mean()
print(f"\nBase rate (test set): {base_rate_test:.3f}")

Comparison table: baseline vs model (same test split, same metric)

Method         Precision@20   Precision@50
baseline              0.400               
baseline              0.400          0.500
model                 0.550          0.560

Base rate (test set): 0.526


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:")
print(importances)

# Look at where the model is wrong
test_results = X_te.copy()
test_results["actual"] = y_te.values
test_results["model_score"] = model_scores_te
test_results["predicted"] = (model_scores_te >= 0.5).astype(int)
wrong = test_results[test_results["actual"] != test_results["predicted"]]
print(f"\n{len(wrong)} wrong predictions out of {len(test_results)} ({len(wrong)/len(test_results):.1%})")
print(wrong[["impressions_90d", "days_since_last_update", "ctr", "avg_position", "actual", "model_score"]].head(5))

Feature importances:
impressions_90d           0.427368
avg_position              0.196541
content_age_days          0.114630
word_count                0.076737
ctr                       0.070830
sessions_90d              0.062128
days_since_last_update    0.051765
dtype: float64

2561 wrong predictions out of 6030 (42.5%)
    impressions_90d  days_since_last_update   ctr  avg_position  actual  \
1             15320                      25  0.05          20.3       1   
13              307                     103  0.00          39.8       0   
26             2426                      13  0.12          30.0       0   
34             3998                       8  0.03           6.4       0   
36              371                      20  1.35           5.4       0   

    model_score  
1      0.451815  
13     0.751790  
26     0.620451  
34     0.651199  
36     0.531857  


Feature importances: impressions_90d (0.43) and avg_position (0.20)
dominate, while days_since_last_update — the feature my baseline rule
was built around — ranks last (0.05). This confirms my Week 4 finding
that staleness alone is a weak/opposite signal for decline; the model is
relying on volume and ranking position instead, which is more plausible.

Error analysis: 42.5% of test predictions were wrong, which is high —
this model is far from reliable on its own. Looking at a few wrong
cases, several show low CTR (0.00-0.12) and low-to-moderate impressions
where the model predicted decline (score > 0.5) but the page wasn't
actually declining, meaning the model may be over-weighting CTR/position
patterns that don't always mean trouble. One row also has ctr = 1.35,
which is impossible for a real CTR and is a known data artifact
consistent with what I noticed in Week 1 — not something I'm treating as
a real signal.

Bottom line: the model beats the baseline at Precision@20 (0.55 vs 0.40)
and Precision@50 (0.56 vs 0.50), both above the base rate (0.526), so
there's real signal here — but a 42.5% overall error rate means this is
still a decision-support tool for a human reviewer, not something to act
on unsupervised.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.